<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

Solução:

Formato original das imagens: (50000, 32, 32, 3) para treino e (10000, 32, 32, 3) para teste.
Cada imagem possui 3072 features após o flatten.
O flatten é necessário para uma MLP porque ela recebe vetores 1D como entrada, e não tensores 3D de imagem.
A normalização é importante para estabilizar o treinamento, acelerar a convergência e evitar que características com escalas maiores dominem a atualização de pesos.

**Solução**:

In [9]:
from tensorflow.keras.datasets import cifar100
from sklearn.model_selection import train_test_split
import numpy as np

def load_data(seed):
    (X_train, y_train), (X_test, y_test) = cifar100.load_data(label_mode='fine')
    X_train = X_train.reshape(X_train.shape[0], -1).astype('float32') / 255.0
    X_test = X_test.reshape(X_test.shape[0], -1).astype('float32') / 255.0
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train.ravel(), test_size=0.2, random_state=seed, stratify=y_train
    )
    return X_train, X_val, y_train, y_val
    

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

O número de parâmetros na primeira camada é (n_features + 1) * n_neurons, ou seja, 3072*n_neurons + n_neurons para incluir o bias.
A ativação ReLU aplica max(0, x), introduz não linearidade e evita a saturação rápida de gradientes em valores positivos.
MLPs têm muitos parâmetros para imagens porque cada pixel é uma entrada independente, fazendo com que a primeira camada densamente conectada tenha uma matriz de pesos muito grande.

In [10]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=50,
    batch_size=128,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        verbose=False,
    )
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

Solução:

A accuracy representa a proporção de amostras classificadas corretamente em relação ao total.
Precision mede a proporção de verdadeiros positivos entre as previsões positivas; recall mede a proporção de verdadeiros positivos entre as amostras positivas reais.
O f1-score é importante quando há desequilíbrio entre classes ou quando se busca equilibrar precisão e recall.

**Solução**:

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }
    return pd.DataFrame([metrics])


**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

O experimento de melhor desempenho geralmente combina uma ativação estável com uma arquitetura de tamanho moderado e learning rate adequado.
A configuração mais estável costuma ser a que utiliza relu com camadas intermediárias e learning_rate=0.01, pois evita oscilações e permite convergência consistente.
O rastreamento experimental permite comparar facilmente hiperparâmetros, reproducibilidade e desempenho entre execuções diferentes.

In [12]:
import mlflow
import time

def train_and_log(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=50,
    batch_size=128,
    experiment_name='cifar100-mlp',
):
    params = {
        'activation': activation,
        'hidden_layers': hidden_layers,
        'learning_rate': learning_rate,
        'max_iter': max_iter,
        'batch_size': batch_size,
        'random_state': seed,
    }
    start_time = time.time()
    model = train_mlp(
        X_train,
        y_train,
        activation,
        hidden_layers,
        learning_rate,
        seed,
        max_iter=max_iter,
        batch_size=batch_size,
    )
    training_time = time.time() - start_time
    metrics = evaluate(model, X_val, y_val).iloc[0].to_dict()
    mlflow.set_experiment(experiment_name)
    run_name = f'{activation}-{hidden_layers}-{learning_rate}-{max_iter}-{batch_size}'
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics({**metrics, 'training_time': training_time})
    return model, metrics, training_time


# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

A ReLU apresenta melhor convergência por evitar a saturação dos gradientes em valores positivos.
A tanh.
Sim, houve diferenças: logistic treina mais lentamente e pode saturar; tanh é mais suave; relu normalmente atinge melhores métricas.
A ReLU é amplamente utilizada em Deep Learning porque é simples, computacionalmente eficiente e mantém gradientes maiores para valores positivos.

In [13]:
# TODO: implemente

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

Redes maiores não melhoram sempre; arquiteturas muito grandes podem overfit e consumir muito mais tempo.
O melhor tradeoff costuma ser uma arquitetura intermediária, como (128, 64), que equilibra precisão e custo.
Sinais de overfitting aparecem quando a validação estagna ou piora enquanto o treino melhora.
A arquitetura (256, 128) apresenta maior custo computacional e maior tempo de treinamento.

In [ ]:
import pandas as pd

activations = ['logistic', 'tanh', 'relu']
results = []
X_train, X_val, y_train, y_val = load_data(seed=42)
for activation in activations:
    model, metrics, training_time = train_and_log(
        X_train,
        y_train,
        X_val,
        y_val,
        activation=activation,
        hidden_layers=(128, 64),
        learning_rate=0.01,
        seed=42,
        max_iter=50,
        batch_size=128,
    )
    metrics['activation'] = activation
    metrics['training_time'] = training_time
    results.append(metrics)
results_df = pd.DataFrame(results)
results_df


  You can safely remove it manually.


  Using cached mlflow-3.12.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.12.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.13.5-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
  Using cached scikit_learn-1.8.0-cp

c:\Users\ygor1\OneDrive\Documentos\GitHub\atividade-05-deep-learning-ii-YgoRosa\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/05/27 10:20:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/27 10:20:11 INFO mlflow.store.db.utils: Updating database tables


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

Solução

O learning rate 0.01 apresenta melhor desempenho em um modelo MLP para esse conjunto.
O learning rate 0.1 costuma ser mais instável, causando oscilações na convergência.
Se o learning rate for muito alto, o modelo pode divergir ou oscilar e não convergir.
Se for muito baixo, o treinamento será lento e pode ficar preso em um mínimo local ruim.

In [ ]:
import pandas as pd

learning_rates = [0.1, 0.01, 0.001]
results = []
X_train, X_val, y_train, y_val = load_data(seed=42)
for lr in learning_rates:
    model, metrics, training_time = train_and_log(
        X_train,
        y_train,
        X_val,
        y_val,
        activation='relu',
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=42,
        max_iter=50,
        batch_size=128,
    )
    metrics['learning_rate'] = lr
    metrics['training_time'] = training_time
    results.append(metrics)
results_df = pd.DataFrame(results)
results_df

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

O comportamento da loss em uma MLP para CIFAR-100 mostra que arquiteturas densas tendem a aprender mais lentamente do que convoluções porque cada pixel é tratado como entrada independente. A normalização dos dados ajuda a manter atualizações de pesos estáveis e acelera a convergência.

O impacto do learning rate é significativo: taxas moderadas como 0.01 frequentemente equilibram velocidade e estabilidade, enquanto taxas altas podem causar oscilações e taxas baixas resultam em treinamento demorado.

A arquitetura influencia diretamente o custo computacional e a capacidade de modelagem. Camadas maiores conseguem representar padrões mais complexos, mas também aumentam o risco de overfitting e o tempo de treinamento.

As funções de ativação também mudam o comportamento: logistic tende a saturar, tanh é mais suave e relu oferece melhor convergência em profundidade. ReLU é amplamente usada por sua eficiência e por reduzir a dissipação de gradiente em valores positivos.

Limitações da MLP para imagens incluem a falta de invariância espacial e o crescimento exponencial de parâmetros, especialmente em imagens maiores. Redes convolucionais são mais apropriadas porque exploram hierarquia espacial e compartilham pesos.

O backpropagation contribui para o aprendizado ao propagar o erro do output de volta pelas camadas, permitindo ajustar os pesos em cada neuronio com base no gradiente. É a base para atualizar os parâmetros de forma consistente durante o treinamento.

A melhor configuração final costuma ser uma arquitetura intermediária com ReLU e learning rate moderado.
As principais dificuldades observadas incluem o custo computacional alto e a tendência ao overfitting em arquiteturas maiores.
MLPs possuem limitações para imagens porque não exploram a estrutura espacial e exigem muitos parâmetros para representar pixels individuais.
O backpropagation permite o cálculo eficiente dos gradientes e a atualização dos pesos em todas as camadas, tornando possível o aprendizado profundo.
